## Day 9 : Random Forest and Ensemble Learning

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

df = pd.read_csv('train.csv')

## Question 1 - Data Prep

In [5]:
X = df[['Pclass','Age','Fare','SibSp','Parch']].copy()
y = df['Survived'].copy()

X['Median_Age'] = X.groupby('Pclass')['Age'].transform('median')
X['Age'].fillna(X['Median_Age'],inplace=True)
print(X.isna().sum())
X.drop(columns=['Median_Age'],inplace=True)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(f"\nX train shape = {X_train.shape}")
print(f"X test shape = {X_test.shape}")


Pclass        0
Age           0
Fare          0
SibSp         0
Parch         0
Median_Age    0
dtype: int64

X train shape = (712, 5)
X test shape = (179, 5)


## Question 2 - Train Random Forest

In [6]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


1. ```n_estimators``` represents the number of trees that will be trained in the Random Forest Model.
2. More trees might help to decrease the variance of the model by taking a decision on average which is not highly affected by single tree biases.

## Question 3 - Evaluate Model

In [9]:
y_pred = model.predict(X_test)
y_true = np.array(y_test)
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

evaluation = pd.DataFrame({
    'Metrics' : ['Accuracy', "Precision",'Recall','F1'],
    "Value" : [acc, precision,recall,f1]
})

evaluation

,Metrics,Value
0,Accuracy,0.759777
1,Precision,0.738462
2,Recall,0.648649
3,F1,0.690647


## Question 4 - Compare Models

| Model | Accuracy | Precision | Recall |
| ----- | -------- | --------- | ------ |
| Logistic Regression | 0.73 | 0.77 | 0.5 |
| Decision Tree | 0.65 | 0.59 | 0.52|
| Random Forest | 0.76 | 0.74 | 0.65|

1. The Random Forest Classifier arguably performs the best out of the 3
2. It shows to have a high accuracy, high precision and an above average recall which is probably from the fact that it consists of a cluster of trees making a decision and then averaging out the decision, thus decreasing variance and ignoring any singular biases.

## Question 5 - Feature Importance

In [13]:
importance = model.feature_importances_
importance_df = pd.DataFrame({
    'Feature' : X.columns,
    'Importance': importance
})

importance_df.sort_values(by='Importance', ascending=False,inplace=True)
importance_df

,Feature,Importance
2,Fare,0.420070
1,Age,0.384880
0,Pclass,0.082588
3,SibSp,0.063942
4,Parch,0.048520


Yes, this agrees with the Decision Tree Importance as well as the correlation analysis.

## Question 6 - Hyperparameter Tuning

In [15]:
# Models
model_A = RandomForestClassifier(n_estimators=10)
model_B = RandomForestClassifier(n_estimators=100)
model_C = RandomForestClassifier(n_estimators=500)

# Model A
model_A.fit(X_train,y_train)
A_pred = model_A.predict(X_test)
acc_A = accuracy_score(np.array(y_test), A_pred)

# Model B
model_B.fit(X_train,y_train)
B_pred = model_B.predict(X_test)
acc_B = accuracy_score(np.array(y_test), B_pred)

# Model C
model_C.fit(X_train,y_train)
C_pred = model_C.predict(X_test)
acc_C = accuracy_score(np.array(y_test), C_pred)

compare_df = pd.DataFrame({
    'Models' : ['Model A','Model B', 'Model C'],
    'Accuracy' : [acc_A,acc_B,acc_C]
})

compare_df

,Models,Accuracy
0,Model A,0.709497
1,Model B,0.731844
2,Model C,0.770950


Well in this case adding more trees is definitely improving the accuracy of our model but i dont think it would be so in all cases. I think it is more dependant on the data and it is possible that adding too many trees could cause overfitting.

## Question 7 - Overfitting Check

In [16]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_acc = accuracy_score(np.array(y_train), train_pred)
test_acc = accuracy_score(np.array(y_test), test_pred)

print(f"Train Acc = {train_acc}\nTest Acc = {test_acc}")

Train Acc = 0.9592696629213483
Test Acc = 0.7597765363128491


Yes it does appear to overfit less than a single decision tree because we are taking an average of 100 decision trees, the model is able to give us better results and performance rather than just a single decision tree that has memorized the train set.

## Question 8 - Probability Predicions

In [31]:
prob_pred = model.predict_proba(X_test)
prob_df = X_test.copy()
prob_df['Prob Yes'] = prob_pred[:,0]
prob_df['Prob No'] = prob_pred[:,1]

prob_df.iloc[:10]

,Pclass,Age,Fare,SibSp,Parch,Prob Yes,Prob No
709,3,24.0,15.2458,1,1,0.518528,0.481472
439,2,31.0,10.5000,0,0,0.925000,0.075000
840,3,20.0,7.9250,0,0,0.875000,0.125000
720,2,6.0,33.0000,0,1,0.060000,0.940000
39,3,14.0,11.2417,1,0,0.530000,0.470000
290,1,26.0,78.8500,0,0,0.110000,0.890000
300,3,24.0,7.7500,0,0,0.560466,0.439534
333,3,16.0,18.0000,2,0,0.924167,0.075833
208,3,16.0,7.7500,0,0,0.473333,0.526667
136,1,19.0,26.2833,0,2,0.230000,0.770000


Probabilities are more useful than simple predictions as probability tells us the chances of an event occuring rather than just whether it will occur or not.

## Question 9 - Data Science Thinking

1. Since we are passing our data through multiple trees the model is comparatively slower and is computationally heavy.
2. For the same reason it has low interpretability as it does not give us a single visual tree structure, a cluster of trees instead which is not very readable.
3. It is best for maximizing predictive accuracy on complex tabular datasets which contain noise and alot of outliers and stuff

# Question 10 - Quant Thinking

Firstly, i need to know the parameters and methods and model used to get this probability forecast.
Secondly, 51% is basically a coin flip with one side being ever so slightly heavier. The odds are still in favour of up but this would still be considered speculation.
The probability of 95% gives a confident answer but I can always question the model for overfitting. Even so, with tuning the probability would not fall very far below 95%, so it still could be trusted more.

In trading systems, probability is useful to decide if a trade should be placed or not. The market is extremely volatile and this knowing if the underlying asset is going to go up or down with high accuracy gives the trader a greater edge to trade while mitigating risk and losses.

## Bonus ML Question

1. Random Forest is usually better because it takes a cluster of multiple trees and averages out a result which decreases variance and ignored singular tree biases.
2. It solves the problem of any bias taken by the deicison tree and decreases variance
3. The tradeoff is interpretability.

---

## DSA Challenge #3

In [36]:
nums = [2,7,11,15]
target = 9

check = {}

for i, num in enumerate(nums):
    compliment = target - num
    if compliment in check:
        print([check[compliment],i])
    check[num] = i

[0, 1]


1. Logic is that i map the number and index into a dict, while iterating through the list I calculate the compliment for each number in the list and check if the compliment is present in the dict and if it is present, it will print the indices of the compliment and the number.
2. O(n)
3. O(n)

## DSA Challenge #4

In [37]:
nums = [1,2,3,4,1]

num_set = set(nums)

if len(nums)!=len(num_set):
    print(True)

True


1. Set establishes a one to one relationship which only keeps a single iteration of a number this discarding duplicates and decreasing the size of the corresponding list/set
2. O(1)